In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

# Load training data
df = pd.read_csv('data/train.csv.gz')

print(f"Dataset Shape: {df.shape}")
print("-" * 50)
print("Data Types & Non-Null Counts:")
df.info()
print("-" * 50)
print("Missing values per column:")
print(df.isnull().sum())

Dataset Shape: (700000, 16)
--------------------------------------------------
Data Types & Non-Null Counts:
<class 'pandas.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 16 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   f0          700000 non-null  float64
 1   f1          700000 non-null  float64
 2   f2          700000 non-null  float64
 3   f3          700000 non-null  float64
 4   f4          700000 non-null  float64
 5   f5          700000 non-null  float64
 6   f6          700000 non-null  float64
 7   f7          700000 non-null  float64
 8   f8          700000 non-null  float64
 9   f9          700000 non-null  float64
 10  f10         700000 non-null  float64
 11  f11         700000 non-null  float64
 12  treatment   700000 non-null  int64  
 13  conversion  700000 non-null  int64  
 14  visit       700000 non-null  int64  
 15  exposure    700000 non-null  int64  
dtypes: float64(12), int64(4)
memor

### Task 1: Dataset Overview & Schema Inspection

#### 1. Data Dimensions & Completeness
* **Record Count:** The dataset contains exactly **700,000 observations** indexed from `0` to `699999`.
* **Missing Values:** All 16 columns have **700,000 non-null entries**, confirming that the dataset contains zero missing values or NaNs that require imputation.

#### 2. Column Structure & Data Types
The schema consists of 16 features categorized into two groups:
* **Pre-Treatment Covariates (`f0`–`f11`):** 12 continuous numerical features typed as `float64`, representing anonymized user features prior to campaign exposure.
* **Experimental Flags & Outcomes:** 4 binary indicator variables typed as `int64`:
  * `treatment`: Assignment indicator to the treatment group ($T = 1$) vs. control holdout ($T = 0$).
  * `exposure`: Impression indicator verifying whether an ad was actually served ($E = 1$) to the user.
  * `visit`: Downstream behavioral target showing whether a user visited the landing page ($Y_v = 1$).
  * `conversion`: Higher-funnel business target showing whether a user made a purchase or conversion action ($Y_c = 1$).

#### 3. Modeling Implications
* **No Imputation Overhead:** With zero missing values across continuous features, pre-processing requires only standard feature scaling rather than missing-data handling.
* **Leakage Prevention:** Features `f0`–`f11` serve as the sole valid feature space for both predictive and causal uplift models. Variables `treatment`, `exposure`, `visit`, and `conversion` are reserved exclusively for assignment, mechanism auditing, and evaluation to prevent post-treatment target leakage.

In [4]:
# Target & Experimental Distributions
target_stats = pd.DataFrame({
    'Metric': [
        'Total Records',
        'Treatment Assigned (T=1)',
        'Control Holdout (T=0)',
        'Actually Exposed (E=1)',
        'Overall Visits (Y_v=1)',
        'Overall Conversions (Y_c=1)'
    ],
    'Count': [
        len(df),
        (df['treatment'] == 1).sum(),
        (df['treatment'] == 0).sum(),
        (df['exposure'] == 1).sum(),
        (df['visit'] == 1).sum(),
        (df['conversion'] == 1).sum()
    ]
})

target_stats['Percentage (%)'] = (target_stats['Count'] / len(df)) * 100
target_stats['Percentage (%)'] = target_stats['Percentage (%)'].round(2)

display(target_stats)

,Metric,Count,Percentage (%)
0,Total Records,700000,100.00
1,Treatment Assigned (T=1),594967,85.00
2,Control Holdout (T=0),105033,15.00
3,Actually Exposed (E=1),21336,3.05
4,Overall Visits (Y_v=1),32857,4.69
5,Overall Conversions (Y_c=1),1987,0.28


### Baseline Distributions & Experimental Breakdown

#### 1. Experimental Split (85/15 Design)
* **Treatment Group ($T=1$):** Accounts for 594,967 records (85.00% of the population).
* **Control Holdout ($T=0$):** Accounts for 105,033 records (15.00% of the population).
* **Design Context:** This is an intentional unbalanced RCT designed to preserve revenue by serving ads to 85% of eligible traffic while keeping a sufficiently powered 15% holdout baseline for causal inference.

#### 2. The Exposure Gap
* **Actual Exposure ($E=1$):** Only 21,336 users (3.05% of the total dataset) were actually served an ad.
* **Delivery Bottleneck:** Even though 85% of users were eligible for ads, the vast majority were never reached due to ad auction loss, low budget, or lack of ad impressions during the window.
* **Causal Implication:** Intention-to-Treat (ITT) estimates using `treatment` measure the overall policy impact across all assigned users, whereas Treatment-on-the-Treated (TOT) would isolate the specific effect among the 3.05% who actually viewed the impression.

#### 3. Target Class Imbalance
* **Visits ($Y_v=1$):** 32,857 users visited the landing page (4.69% base rate).
* **Conversions ($Y_c=1$):** 1,987 users converted (0.28% base rate).
* **Modeling Impact:** Conversions represent a rare-event target (<0.3%). Standard log-loss and accuracy metrics will default to predicting non-events, making rank-order evaluations (such as AUUC and decile uplift tables) essential over uncalibrated binary classification accuracy.

In [5]:
# Randomization Check (Covariate Balance)
feature_cols = [f'f{i}' for i in range(12)]

trt = df[df['treatment'] == 1][feature_cols]
ctrl = df[df['treatment'] == 0][feature_cols]

balance_df = pd.DataFrame({
    'Treated Mean': trt.mean(),
    'Treated Std': trt.std(),
    'Control Mean': ctrl.mean(),
    'Control Std': ctrl.std()
})

# Standardized Mean Difference (SMD)
pooled_sd = np.sqrt((balance_df['Treated Std']**2 + balance_df['Control Std']**2) / 2)
balance_df['SMD'] = (balance_df['Treated Mean'] - balance_df['Control Mean']) / pooled_sd

# Rule of thumb: |SMD| > 0.1 indicates potential imbalance
balance_df['Balanced (|SMD| < 0.1)'] = balance_df['SMD'].abs() < 0.1

display(balance_df.round(4))
print(f"Max absolute SMD across all features: {balance_df['SMD'].abs().max():.5f}")

,Treated Mean,Treated Std,Control Mean,Control Std,SMD,Balanced (|SMD| < 0.1)
f0,19.6078,5.3770,19.6610,5.3806,-0.0099,True
f1,10.0702,0.1057,10.0683,0.0932,0.0193,True
f2,8.4463,0.2992,8.4446,0.2994,0.0057,True
f3,4.1717,1.3476,4.2241,1.2504,-0.0403,True
f4,10.3390,0.3413,10.3372,0.3414,0.0051,True
f5,4.0276,0.4311,4.0377,0.4016,-0.0242,True
f6,-4.1761,4.5899,-4.0248,4.4510,-0.0335,True
f7,5.1047,1.2112,5.0824,1.1663,0.0188,True
f8,3.9334,0.0568,3.9347,0.0556,-0.0235,True
f9,16.0566,7.0602,15.9201,6.8532,0.0196,True


Max absolute SMD across all features: 0.04033
